<a href="https://colab.research.google.com/github/MdMostafizurRahaman/Machine-Learning/blob/main/Exercise%20-2/logistic-regression-lab-exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚯 Lecture 11 Lab: Logistic regression and spam detection

<img src="https://github.com/joshuagrossman/mse125-labs-public/blob/main/hw5/img/spam-email.png?raw=1" alt= “spam-email” width="500" />

## ✅ Setup and data import
In this lab, we will work with a [classic dataset](https://archive.ics.uci.edu/dataset/94/spambase) of 4,601 emails classified as spam or not spam.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Use three digits past the decimal point
pd.set_option('display.precision', 3)

# Format plots with a white background
sns.set_style('whitegrid')

# Increase the default text size of plots
plt.rcParams.update({'font.size': 20})

# Increase the default plot width and height
plt.rcParams['figure.figsize'] = (12, 8)

# Read in the data
spam = pd.read_csv('spam.csv')

# peek at 10 random rows
spam.sample(10)

,make,address,all,3d,our,over,remove,internet,order,mail,...,char_semicolon,char_left_paren,char_left_bracket,char_exclamation,char_dollar,char_pound,capital_run_length_average,capital_run_length_longest,capital_run_length_total,is_spam
1980,0.00,0.0,0.00,0.0,0.00,0.00,0.0,0.00,0.0,1.23,...,0.000,0.796,0.000,0.000,0.0,0.0,2.142,8,60,0
2282,0.00,0.0,0.00,0.0,0.00,0.00,0.0,0.00,0.0,0.00,...,0.325,0.000,0.000,0.000,0.0,0.0,1.000,1,14,0
4117,0.00,0.0,0.00,0.0,0.00,0.00,0.0,0.00,0.0,0.00,...,0.000,0.000,0.000,0.259,0.0,0.0,1.681,12,37,0
3908,0.00,0.0,0.33,0.0,0.33,0.00,0.0,0.00,0.0,0.00,...,0.000,0.175,0.058,0.000,0.0,0.0,2.068,12,120,0
1189,0.00,0.0,0.00,0.0,0.00,0.00,0.0,0.00,0.0,0.00,...,0.000,0.000,0.000,1.473,0.0,0.0,4.071,29,114,1
3416,0.00,0.0,0.00,0.0,0.00,0.49,0.0,0.49,0.0,0.00,...,0.000,0.228,0.000,0.000,0.0,0.0,1.962,5,106,0
2466,0.00,0.0,0.00,0.0,0.00,0.00,0.0,0.00,0.0,0.00,...,0.000,0.000,0.000,0.000,0.0,0.0,1.785,6,25,0
1197,0.67,0.0,0.67,0.0,0.00,0.00,0.0,0.00,0.0,0.67,...,0.000,0.000,0.000,2.413,0.0,0.0,3.384,37,132,1
3951,0.00,0.0,2.08,0.0,0.00,0.00,0.0,0.00,0.0,0.00,...,0.000,0.000,0.000,0.000,0.0,0.0,1.000,1,11,0
4188,0.00,0.0,0.00,0.0,0.00,0.00,0.0,0.00,0.0,0.00,...,0.000,0.102,0.000,0.000,0.0,0.0,3.400,51,119,0


## ♨️ Warm up

How many emails are in the database?

What fraction of the emails in the database are spam?

Which email contains the highest percentage of words matching "money"? What percentage of words in that email match "money"?

In [3]:
# How many emails are in the database?
print(f"Number of emails: {len(spam)}")

# What fraction of the emails in the database are spam?
spam_fraction = spam['is_spam'].mean()
print(f"Fraction of emails that are spam: {spam_fraction}")

# Which email contains the highest percentage of words matching "money"? What percentage?
max_money_idx = spam['money'].idxmax()
max_money_pct = spam.loc[max_money_idx, 'money']
print(f"Email with highest 'money' percentage: index {max_money_idx}, percentage: {max_money_pct}")

Number of emails: 4601
Fraction of emails that are spam: 0.39404477287546186
Email with highest 'money' percentage: index 545, percentage: 12.5


## 🎲 Linear probability models (LPMs)

Fit a linear regression model to the spam data with the `lm` function.

Use the following covariates to predict the likelihood that an email is spam:
- `char_dollar`
- `credit`
- `money`
- `re`

How would you interpret the model coefficients for the intercept and for `char_dollar`?

- Note: `char_dollar` represents the percentage of characters in the email that match `$`.

In [4]:
from sklearn.linear_model import LinearRegression

X = spam[['char_dollar', 'credit', 'money', 're']]
y = spam['is_spam']

lpm_model = LinearRegression()
lpm_model.fit(X, y)

print("Linear Probability Model Coefficients:")
print(f"Intercept: {lpm_model.intercept_:.3f}")
for name, coef in zip(X.columns, lpm_model.coef_):
    print(f"{name}: {coef:.3f}")

# Interpretation:
# Intercept: The predicted probability of an email being spam when all predictors (char_dollar, credit, money, re) are zero.
# char_dollar: The change in predicted probability of being spam for a 1% increase in the percentage of characters that are '$'.

Linear Probability Model Coefficients:
Intercept: 0.335
char_dollar: 0.586
credit: 0.158
money: 0.188
re: -0.054


Using your linear probability model and the `predict` function, predict the in-sample probability that each email is spam.

What is the smallest predicted probability? The largest? Do you notice any issues with these predictions?

In [5]:
predictions = lpm_model.predict(X)

print(f"Smallest predicted probability: {predictions.min():.3f}")
print(f"Largest predicted probability: {predictions.max():.3f}")

# Issues: The predicted probabilities can be less than 0 or greater than 1, which are invalid probabilities.
# For example, some predictions are negative, meaning impossible probabilities.

Smallest predicted probability: -0.813
Largest predicted probability: 3.849


## 🎰 Odds functions

Write two functions:
- A function to convert probabilities to odds.
- A function to convert odds to probabilities

Test your functions by making sure that 2:1 odds returns a 2/3 probability, and vice versa.

Finally, suppose my probability of winning is 60%. If I double my odds of winning, what is my new probability of winning?

In [6]:
def prob_to_odds(p):
    return p / (1 - p)

def odds_to_prob(o):
    return o / (1 + o)

# Test: 2:1 odds should be 2/3 probability
odds_test = prob_to_odds(2/3)
print(f"Odds for probability 2/3: {odds_test:.3f}")  # Should be 2.0

prob_test = odds_to_prob(2)
print(f"Probability for 2:1 odds: {prob_test:.3f}")  # Should be 2/3

# Suppose probability of winning is 60%. If I double my odds, what is new probability?
current_prob = 0.6
current_odds = prob_to_odds(current_prob)
new_odds = current_odds * 2
new_prob = odds_to_prob(new_odds)
print(f"New probability after doubling odds: {new_prob:.3f}")

Odds for probability 2/3: 2.000
Probability for 2:1 odds: 0.667
New probability after doubling odds: 0.750


## 🪙 Fitting a logistic regression model

We can fit a logistic regression model with the same covariates as above with the following code:

In [7]:
import statsmodels.api as sm

logit_model = sm.Logit(y, sm.add_constant(X))
result = logit_model.fit()
print(result.summary())

Optimization terminated successfully.
         Current function value: 0.481178
         Iterations 8
                           Logit Regression Results                           
Dep. Variable:                is_spam   No. Observations:                 4601
Model:                          Logit   Df Residuals:                     4596
Method:                           MLE   Df Model:                            4
Date:                Tue, 18 Nov 2025   Pseudo R-squ.:                  0.2824
Time:                        11:08:00   Log-Likelihood:                -2213.9
converged:                       True   LL-Null:                       -3085.1
Covariance Type:            nonrobust   LLR p-value:                     0.000
                  coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------
const          -1.0666      0.043    -24.680      0.000      -1.151      -0.982
char_dollar    11.8176    

Interpret the intercept and `money` coefficients for the logistic regression model three different ways:
1. On the log odds scale
2. On the odds scale (by exponentiating the coefficients)
3. On the probability scale (using either the odds functions you wrote, or the divide by 4 trick).

Tip: Use the `coef` function to extract coefficients from the model.

In [8]:
coef = result.params

print("Logistic Regression Coefficients:")
for name, c in coef.items():
    print(f"{name}: {c:.3f}")

# 1. On the log odds scale:
# Intercept: The log odds of an email being spam when all predictors are zero.
# money: The change in log odds of being spam for a 1 unit increase in the 'money' variable.

# 2. On the odds scale (by exponentiating):
print("\nExponentiated coefficients (odds scale):")
for name, c in coef.items():
    print(f"{name}: {np.exp(c):.3f}")
# Intercept: The odds of being spam when predictors are zero.
# money: The multiplier for odds of being spam for a 1 unit increase in 'money'.

# 3. On the probability scale (using divide by 4 trick for small changes):
print("\nApproximate change in probability (divide by 4):")
for name, c in coef.items():
    print(f"{name}: {c/4:.3f}")
# This approximates the change in probability for a small increase in the predictor.

Logistic Regression Coefficients:
const: -1.067
char_dollar: 11.818
credit: 2.312
money: 1.993
re: -0.776

Exponentiated coefficients (odds scale):
const: 0.344
char_dollar: 135613.881
credit: 10.094
money: 7.340
re: 0.460

Approximate change in probability (divide by 4):
const: -0.267
char_dollar: 2.954
credit: 0.578
money: 0.498
re: -0.194
